In [ ]:
# Cell 1: Verify CUDA environment and install PyTorch Geometric with dependencies
import os
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device name:     {torch.cuda.get_device_name(0)}")
    print(f"CUDA version:    {torch.version.cuda}")

# Export current PyTorch + CUDA version to environment variable (e.g., 2.5.1+cu121)
os.environ['TORCH'] = torch.__version__

# Install pre-built wheels for torch-scatter and torch-sparse matching Colab's CUDA version
!pip install -q torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-${TORCH}.html

# Install main torch-geometric package
!pip install -q torch-geometric

PyTorch version: 2.11.0+cu130
CUDA available:  True
Device name:     NVIDIA RTX PRO 6000 Blackwell Server Edition
CUDA version:    13.0


In [ ]:
# Cell 2: Mount Google Drive to prepare for loading the Elliptic Dataset
from google.colab import drive
import os

drive.mount('/content/drive')

# Expected path to the Elliptic dataset directory on Google Drive
ELLIPTIC_DATA_DIR = '/content/drive/MyDrive/Explainable-AML-GAT-ResNet/data'

if os.path.exists(ELLIPTIC_DATA_DIR):
    print(f"[INFO] Dataset directory found at: {ELLIPTIC_DATA_DIR}")
    print("Files in directory:", os.listdir(ELLIPTIC_DATA_DIR))
else:
    print("[INFO] Google Drive mounted successfully at '/content/drive'!")
    print(f"[WARN] Directory '{ELLIPTIC_DATA_DIR}' does not exist yet or is located at a different path.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[INFO] Dataset directory found at: /content/drive/MyDrive/Explainable-AML-GAT-ResNet/data
Files in directory: []


In [ ]:
# Cell 3: Verify PyTorch Geometric installation with a dummy Data object
import torch
import torch_geometric
import torch_scatter
import torch_sparse
from torch_geometric.data import Data

print(f"PyTorch Geometric version: {torch_geometric.__version__}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. Initialize dummy node feature matrix (x):
# Simulate 4 transaction nodes, each with 166 features (matching the Elliptic Dataset)
x = torch.randn((4, 166), dtype=torch.float)

# 2. Initialize directed edge list (edge_index) in COO format [2, num_edges]:
# Simulate fund flows between transactions: 0 -> 1, 1 -> 2, 2 -> 3, 0 -> 3
edge_index = torch.tensor([
    [0, 1, 2, 0],  # Source nodes
    [1, 2, 3, 3]   # Target nodes
], dtype=torch.long)

# 3. Create PyG Data object and move it to GPU (if available)
sample_graph = Data(x=x, edge_index=edge_index).to(device)

# 4. Validate graph data structure integrity
sample_graph.validate(raise_on_error=True)

# 5. Print verification summary to terminal
print("\n=== PYG DATA OBJECT VERIFICATION ===")
print(sample_graph)
print(f"- Storage device:          {sample_graph.x.device}")
print(f"- Number of nodes:         {sample_graph.num_nodes}")
print(f"- Number of edges:         {sample_graph.num_edges}")
print(f"- Node feature dimension:  {sample_graph.num_node_features}")
print(f"- Is directed graph:       {sample_graph.is_directed()}")
print(f"- Has isolated nodes:      {sample_graph.has_isolated_nodes()}")
print("[SUCCESS] PyTorch Geometric environment is ready!")

PyTorch Geometric version: 2.8.1

=== PYG DATA OBJECT VERIFICATION ===
Data(x=[4, 166], edge_index=[2, 4])
- Storage device:          cuda:0
- Number of nodes:         4
- Number of edges:         4
- Node feature dimension:  166
- Is directed graph:       True
- Has isolated nodes:      False
[SUCCESS] PyTorch Geometric environment is ready!
